# DTW Trajectory Analysis - Cohort Runner

This notebook runs **DTW Trajectory Analysis** for all configured cohorts and age bands.

- **Script**: `10d_dtw_dashboard_visual/create_dtw_features.py`
- **Purpose**: Extract dynamic time warping-based trajectory features
- **Outputs**: DTW trajectory features added to model data

## Features

✅ **Dynamic cohort selection** - Configure which cohorts/age bands to run  
✅ **Idempotent** - Automatically skips completed analyses  
✅ **S3 integration** - Results automatically synced to S3

## Usage

1. Configure cohorts and age bands in the configuration cell below
2. Run the execution cell to process all combinations
3. Results are saved locally and synced to S3 automatically

In [ ]:
# Environment Setup

import os
import sys
from pathlib import Path

def resolve_python_bin() -> Path:
    env_bin = os.environ.get("COHORT_RUNNER_PYTHON")
    if env_bin:
        return Path(env_bin)
    return Path(sys.executable)

PYTHON_BIN = resolve_python_bin()
print(f"[INFO] Using Python binary: {PYTHON_BIN}")

def resolve_project_root() -> Path:
    if "__file__" in globals():
        root = Path(__file__).resolve().parents[1]
    else:
        notebook_path = Path(os.getcwd()).resolve()
        if notebook_path.name == "10d_dtw_dashboard_visual":
            root = notebook_path.parent
        else:
            root = notebook_path
            for parent in notebook_path.parents:
                if parent.name == "pgx-analysis":
                    root = parent
                    break
    # Ensure we use repo root (must contain 4_model_data) so subprocess cwd is correct
    if not (root / "4_model_data").exists():
        for parent in root.parents:
            if (parent / "4_model_data").exists():
                return parent
    return root

PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from py_helpers.constants import COHORT_NAMES, AGE_BANDS

## Configuration

Select which cohorts and age bands to process. Leave empty lists to process all.

In [ ]:
# Configuration: Select cohorts and age bands to process
COHORTS_TO_RUN = []  # e.g., ['opioid_ed'] or [] for all
AGE_BANDS_TO_RUN = []  # e.g., ['0-12', '13-24'] or [] for all

if not COHORTS_TO_RUN:
    COHORTS_TO_RUN = COHORT_NAMES.copy()
if not AGE_BANDS_TO_RUN:
    AGE_BANDS_TO_RUN = AGE_BANDS.copy()

print(f"Will process {len(COHORTS_TO_RUN)} cohort(s) and {len(AGE_BANDS_TO_RUN)} age band(s)")
print(f"Cohorts: {COHORTS_TO_RUN}")
print(f"Age bands: {AGE_BANDS_TO_RUN}")

## Run DTW Trajectory Analysis

In [ ]:
# Run DTW Trajectory Analysis for all configured cohort/age band combinations
# DTW has two steps: 1) create_dtw_features.py, 2) add_dtw_features_to_model_data.py

import subprocess

FAIL_FAST = True  # Stop on first failure; set to False to continue on errors

combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Running DTW Trajectory Analysis for {len(combinations)} combinations...")
print("=" * 80)

for cohort_name, age_band in combinations:
    print(f"\n[DTW] Starting: {cohort_name} / {age_band}")
    print("-" * 80)
    
    # Step 1: Create DTW features
    print(f"[DTW] Step 1/2: Creating DTW features...")
    result1 = subprocess.run(
        [str(PYTHON_BIN), "10d_dtw_dashboard_visual/create_dtw_features.py",
         "--cohort", cohort_name,
         "--age_band", age_band],
        cwd=PROJECT_ROOT,
    )
    
    if result1.returncode != 0:
        msg = f"[DTW] Step 1 FAILED ({result1.returncode}): {cohort_name} / {age_band}"
        print(msg)
        if FAIL_FAST:
            raise RuntimeError(msg)
        continue
    
    # Step 2: Add DTW features to model data
    print(f"[DTW] Step 2/2: Adding DTW features to model data...")
    result2 = subprocess.run(
        [str(PYTHON_BIN), "10d_dtw_dashboard_visual/add_dtw_features_to_model_data.py",
         "--cohort-name", cohort_name,
         "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    
    if result2.returncode == 0:
        print(f"[DTW] COMPLETED: {cohort_name} / {age_band}")
    else:
        msg = f"[DTW] Step 2 FAILED ({result2.returncode}): {cohort_name} / {age_band}"
        print(msg)
        if FAIL_FAST:
            raise RuntimeError(msg)

print("\n" + "=" * 80)
print("All DTW Trajectory Analysis completed.")
print("=" * 80)